## Setup Environment & Code

In [ ]:
from google.colab import drive
import os, sys, shutil
from pathlib import Path

# --- ⚙️ CONFIGURATION --- 
REPO_URL = "https://github.com/paranietharan/pothole-detection.git"
PROJECT_NAME = "pothole-detection"
DRIVE_SAVE_PATH = Path("/content/drive/MyDrive/pothole_project/processed_dataset")

LOCAL_RAW = Path("/content/raw_data")
LOCAL_WORK = Path("/content/workspace")
LOCAL_OUT = Path("/content/yolo_dataset")

def setup_colab():
    """Mounts drive and clones the repository."""
    # 1. Mount Drive for saving results
    drive.mount('/content/drive')
    DRIVE_SAVE_PATH.mkdir(parents=True, exist_ok=True)

    # 2. Clone Repository
    %cd /content
    if os.path.exists(PROJECT_NAME): shutil.rmtree(PROJECT_NAME)
    print(f"📡 Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    
    # 3. Add Cloned Repo to Path
    SCRIPTS_DIR = Path(f"/content/{PROJECT_NAME}")
    if str(SCRIPTS_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPTS_DIR))
    
    # 4. Prepare local workspace
    for p in [LOCAL_RAW, LOCAL_WORK, LOCAL_OUT]:
        if p.exists(): shutil.rmtree(p)
        p.mkdir(parents=True)
    
    print("✅ Setup Complete. Scripts and workspace ready.")

In [ ]:
!pip install -q ultralytics imagehash python-dotenv kaggle opencv-python
setup_colab()

## Dataset Acquisition

In [ ]:
# --- 🔑 KAGGLE CREDENTIALS --- 
os.environ['KAGGLE_USERNAME'] = "your_username" 
os.environ['KAGGLE_KEY'] = "your_api_key"

import utils.downloader
DATASET_SLUG = "aliabdelmenam/rdd-2022"

print(f"⏬ Downloading {DATASET_SLUG}...")
success = utils.downloader.download_dataset(DATASET_SLUG, LOCAL_RAW)
if success:
    !ls -R {str(LOCAL_RAW)} | head -n 10

## Preprocessing Execution

In [ ]:
# Configure Environment for your config.py
os.environ['RDD2022_ROOT'] = str(LOCAL_RAW / "rdd2022") 
os.environ['WORK_DIR'] = str(LOCAL_WORK)
os.environ['YOLO_DATASET_ROOT'] = str(LOCAL_OUT)
os.environ['LOG_DIR'] = str(LOCAL_WORK / "logs")

print("🚀 Starting Preprocessing Pipeline...")
try:
    # Import the main script from the cloned repo
    import preprocess
    preprocess.main()
    print("✅ Processing complete.")
except Exception as e:
    print(f"❌ Pipeline Error: {e}")

## Save to Drive

In [ ]:
# 1. Generate data.yaml (pointing to Drive path)
yaml_content = f"path: {str(DRIVE_SAVE_PATH)}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: Pothole"
with open(LOCAL_OUT / "data.yaml", "w") as f:
    f.write(yaml_content)

# 2. Sync results from local Colab to persistent Google Drive
print(f"📤 Syncing to Drive: {DRIVE_SAVE_PATH}...")
!rsync -ah --progress {str(LOCAL_OUT)}/ {str(DRIVE_SAVE_PATH)}/
print("\n✨ Pipeline Complete! Your model-ready dataset is in Google Drive.")